<a href="https://colab.research.google.com/github/AtheerMahir/intermediate-java-5-projects-5954093/blob/main/Week_04_Transcribe_in_Google_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AI Atheer Mind — Week 04 multilingual lecture transcription

This notebook generates **machine-generated timestamped captions** from your original lecture recording. Its outputs are **not** a verified slide-timing map or a final lecture presentation.

**Privacy:** You will explicitly upload the recording into **Google Colab**, a Google-operated cloud session. Do not run this cloud workflow unless you are comfortable processing the recording there and have permission to do so. For private/offline processing, use the companion `transcribe_lecture.py` on your own computer. The notebook neither sends files to paid speech APIs nor shares them through other services. On first run it downloads free open-source model weights.

**Steps:** 1. Run installation; 2. upload the M4A; 3. install the bundled script into the Colab session; 4. **test a real 30-second sample**; 5. run full-lecture transcription; 6. download the transcript review ZIP and upload it back to this conversation.

In Colab, consider **Runtime → Change runtime type → T4 GPU** if a free GPU is offered. Availability, quota, internet access, and transcription accuracy cannot be guaranteed. `small` is an English/Arabic-capable **multilingual** model. Choose `medium` for potentially better accuracy if resources allow; avoid `*.en` models for Arabic.


In [9]:
# Step 1 — install dependency
import sys, subprocess, shutil
assert shutil.which("ffmpeg") and shutil.which("ffprobe"), "FFmpeg and ffprobe required."
# Installing faster-whisper normally without pinning av to avoid source build failures
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "faster-whisper"], check=True)
print("Dependencies installed successfully. We will patch PyAV compatibility in Step 3.")

Dependencies installed successfully. We will patch PyAV compatibility in Step 3.


In [2]:
# Step 2 — upload your OWN original lecture recording into this Colab session.
from google.colab import files
uploaded = files.upload()
if not uploaded:
    raise RuntimeError("No audio/video uploaded")
audio_path = "/content/" + next(iter(uploaded))
print("Audio available for transcription:", audio_path)


Saving Week_04_24m_Rescuing_Williwonk_s_Chocolates_with_Systems_Analys.m4a to Week_04_24m_Rescuing_Williwonk_s_Chocolates_with_Systems_Analys.m4a
Audio available for transcription: /content/Week_04_24m_Rescuing_Williwonk_s_Chocolates_with_Systems_Analys.m4a


In [10]:
# Step 3 — write the same complete local Python script inside the notebook with a PyAV monkeypatch.
from pathlib import Path
Path('/content/transcribe_lecture.py').write_text('#!/usr/bin/env python3\n"""\nAI Atheer Mind - free local / Google Colab timestamped transcription runner.\nSource audio never leaves the selected runtime through this script.\nRequires: Python 3.9+, FFmpeg/FFprobe, faster-whisper (downloads model weights\non first use). The model\u0027s transcripts are MACHINE GENERATED and MUST be reviewed.\n"""\nfrom __future__ import annotations\n\n# MONKEYPATCH: Resolve PyAV v13+ TypeError: open() got an unexpected keyword argument \u0027metadata_errors\u0027\ntry:\n    import av\n    original_av_open = av.open\n    def patched_av_open(*args, **kwargs):\n        kwargs.pop("metadata_errors", None)\n        return original_av_open(*args, **kwargs)\n    av.open = patched_av_open\nexcept ImportError:\n    pass\n\nimport argparse\nimport csv\nimport datetime as dt\nimport hashlib\nimport json\nimport os\nimport platform\nimport re\nimport shutil\nimport subprocess\nimport sys\nimport zipfile\nfrom pathlib import Path\n\n\ndef run_cmd(args):\n    result = subprocess.run(args, capture_output=True, text=True)\n    if result.returncode:\n        raise RuntimeError(\n            "Command failed: " + " ".join(map(str, args)) +\n            "\\n" + (result.stderr or result.stdout)[-1200:]\n        )\n    return result.stdout\n\n\ndef probe(path: Path):\n    data = json.loads(run_cmd([\n        "ffprobe", "-v", "error", "-show_entries",\n        "format=duration,format_name:stream=codec_type,codec_name,sample_rate,channels",\n        "-of", "json", str(path)\n    ]))\n    streams = data.get("streams", [])\n    if not any(s.get("codec_type") == "audio" for s in streams):\n        raise RuntimeError("The input contains no detectable audio stream.")\n    duration = float(data["format"]["duration"])\n    if duration <= 0:\n        raise RuntimeError("Invalid or zero audio duration.")\n    return {"duration_seconds": duration,\n            "format": data.get("format", {}).get("format_name"),\n            "streams": streams}\n\n\ndef digest(path: Path):\n    sha = hashlib.sha256()\n    with path.open("rb") as f:\n        for chunk in iter(lambda: f.read(4 * 1024 * 1024), b""):\n            sha.update(chunk)\n    return sha.hexdigest()\n\n\ndef timestamp(seconds, srt=False):\n    millis = max(0, round(float(seconds) * 1000))\n    ms = millis % 1000\n    sec = (millis // 1000) % 60\n    minute = (millis // 60000) % 60\n    hour = millis // 3600000\n    separator = "," if srt else "."\n    return f"{hour:02d}:{minute:02d}:{sec:02d}{separator}{ms:03d}"\n\n\ndef flags_for(seg):\n    problems = []\n    lp = getattr(seg, "avg_logprob", None)\n    ns = getattr(seg, "no_speech_prob", None)\n    cr = getattr(seg, "compression_ratio", None)\n    if lp is not None and lp < -0.75:\n        problems.append("low_mean_log_probability")\n    if ns is not None and ns > 0.65:\n        problems.append("high_no_speech_probability")\n    if cr is not None and cr > 2.4:\n        problems.append("high_compression_ratio_check_for_repetition")\n    if not getattr(seg, "text", "").strip():\n        problems.append("empty_text")\n    if float(seg.end) < float(seg.start):\n        problems.append("invalid_time_order")\n    return problems\n\n\ndef record(seg, cue_id):\n    words = []\n    for word in (getattr(seg, "words", None) or []):\n        words.append({\n            "start_seconds": getattr(word, "start", None),\n            "end_seconds": getattr(word, "end", None),\n            "text": getattr(word, "word", ""),\n            "probability": getattr(word, "probability", None)\n        })\n    start = float(seg.start)\n    end = float(seg.end)\n    warnings = flags_for(seg)\n    return {\n        "cue_id": cue_id,\n        "start_seconds": start,\n        "end_seconds": end,\n        "start_timestamp": timestamp(start),\n        "end_timestamp": timestamp(end),\n        "text": str(getattr(seg, "text", "")).strip(),\n        "avg_logprob": getattr(seg, "avg_logprob", None),\n        "no_speech_prob": getattr(seg, "no_speech_prob", None),\n        "compression_ratio": getattr(seg, "compression_ratio", None),\n        "uncertainty_flags": warnings,\n        "review_required": bool(warnings),\n        "words": words\n    }\n\n\ndef dump_json(path, data):\n    path.write_text(json.dumps(data, ensure_ascii=False, indent=2, default=str) + "\\n",\n                    encoding="utf-8")\n\n\ndef language_arg(s):\n    return None if s.lower() in ("auto", "none") else s\n\n\ndef transcribe(model, audio_path, language, word_timestamps=False, beam_size=5):\n    # Keep original times. VAD off to avoid suppressing unusual or quiet speech.\n    return model.transcribe(\n        str(audio_path), language=language_arg(language),\n        beam_size=beam_size, word_timestamps=word_timestamps,\n        vad_filter=False, condition_on_previous_text=True\n    )\n\n\ndef write_outputs(outdir, source, segments, metadata):\n    vtt = ["WEBVTT", ""]\n    srt = []\n    txt = []\n    for row in segments:\n        idx = row["cue_id"]\n        start, end, sentence = row["start_seconds"], row["end_seconds"], row["text"]\n        vtt.extend([str(idx), f"{timestamp(start)} --> {timestamp(end)}", sentence, ""])\n        srt.extend([str(idx), f"{timestamp(start, True)} --> {timestamp(end, True)}",\n                    sentence, ""])\n        txt.append(f\u0027[{timestamp(start)} --> {timestamp(end)}] {sentence}\u0027)\n    (outdir / "transcript.vtt").write_text("\\n".join(vtt) + "\\n", encoding="utf-8")\n    (outdir / "transcript.srt").write_text("\\n".join(srt) + "\\n", encoding="utf-8")\n    (outdir / "transcript.txt").write_text("\\n".join(txt) + "\\n", encoding="utf-8")\n    dump_json(outdir / "transcript.json", {"metadata": metadata, "cues": segments})\n    fields = ["cue_id", "start_timestamp", "end_timestamp", "start_seconds",\n              "end_seconds", "text", "avg_logprob", "no_speech_prob",\n              "compression_ratio", "review_required", "uncertainty_flags"]\n    with (outdir / "transcript.csv").open("w", encoding="utf-8-sig", newline="") as f:\n        w = csv.DictWriter(f, fieldnames=fields)\n        w.writeheader()\n        for row in segments:\n            entry = {key: row.get(key) for key in fields}\n            entry["uncertainty_flags"] = "; ".join(row["uncertainty_flags"])\n            w.writerow(entry)\n    flagged = [f\u0027{r["cue_id"]}: {", ".join(r["uncertainty_flags"])}\u0027\n               for r in segments if r["review_required"]]\n    notes = [\n        "AI ATHEER MIND - TRANSCRIPTION UNCERTAINTY & REVIEW",\n        "This is MACHINE transcription, not a human-verified transcript.",\n        "Confidence heuristics are not guarantees. Segments without flags can still contain errors.",\n        "Check names, proper nouns, case-study terminology, numbers, language switches,",\n        "spoken examples, missing/duplicated phrases and transitions against the recording.",\n        "Cue boundaries are estimated by ASR, not certified exact teaching-slide boundaries.",\n        "Audio-duration gaps and silence are NOT represented as transcript cues.",\n        "Word timestamps (if present in JSON) also require listening/alignment review.",\n        f"Detected language (full recording): {metadata.get(\u0027detected_language\u0027)}",\n        f"Flagged cues: {len(flagged)} of {len(segments)}",\n        "",\n        "FLAGGED CUES:",\n        *(flagged or ["None flagged automatically; manual listening is still essential."]),\n        ""\n    ]\n    (outdir / "uncertainty_notes.txt").write_text("\\n".join(notes), encoding="utf-8")\n    dump_json(outdir / "run_metadata.json", metadata)\n    (outdir / "README_FIRST.txt").write_text(\n        "AI ATHEER MIND / WEEK 04 - TRANSCRIPT REVIEW PACKAGE\\n\\n"\n        "Please upload this whole ZIP into the AI Atheer Mind conversation.\\n"\n        "It contains transcript.vtt, transcript.srt, transcript.txt,\\n"\n        "transcript.json (including available word timestamps), transcript.csv,\\n"\n        "run_metadata.json, uncertainty_notes.txt, sample_sanity.json and manifest.json.\\n\\n"\n        "The source audio is NOT copied into this ZIP.\\n"\n        "TRANSCRIPTION IS UNREVIEWED: speech accuracy and conceptual slide\\n"\n        "boundaries cannot be certified merely from these timestamps.\\n",\n        encoding="utf-8"\n    )\n    output_files = sorted(p for p in outdir.iterdir() if p.is_file() and p.name != "manifest.json")\n    manifest = {\n        "package_type": "machine_transcription_for_review_not_finished_slides",\n        "source_filename": source.name,\n        "files": [\n            {"file": x.name, "size_bytes": x.stat().st_size, "sha256": digest(x)}\n            for x in output_files\n        ]\n    }\n    dump_json(outdir / "manifest.json", manifest)\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--input", required=True, type=Path,\n                        help="Source recording (M4A/MP3/WAV/MP4/MOV/etc.)")\n    parser.add_argument("--output-dir", type=Path, default=Path("atheer_transcripts"),\n                        help="Where to write the extracted transcripts and ZIP")\n    parser.add_argument("--model", default="small",\n                        help="Multilingual model: small (default), medium, large-v3, etc. Do NOT use .en models for Arabic.")\n    parser.add_argument("--language", default="auto",\n                        help="auto, en, ar, or another language code. \u0027auto\u0027 detects source language.")\n    parser.add_argument("--device", default="auto", choices=["auto", "cpu", "cuda"])\n    parser.add_argument("--compute-type", default="auto")\n    parser.add_argument("--sample-start", type=float, default=60.0,\n                        help="Start time in seconds for REAL-recording sample test")\n    parser.add_argument("--sample-duration", type=float, default=30.0)\n    parser.add_argument("--sample-only", action="store_true",\n                        help="Test a real 30s sample and stop without full-lecture inference.")\n    args = parser.parse_args()\n\n    src = args.input.expanduser().resolve()\n    if not src.is_file():\n        parser.error(f"No input file at: {src}")\n    if not shutil.which("ffmpeg") or not shutil.which("ffprobe"):\n        parser.error("FFmpeg and ffprobe are required on PATH.")\n    meta_probe = probe(src)\n    duration = meta_probe["duration_seconds"]\n    outdir = args.output_dir.resolve() / re.sub(r"[^\\w.-]+", "_", src.stem)\n    outdir.mkdir(parents=True, exist_ok=True)\n    try:\n        # Inject monkeypatch into currently running namespace just in case\n        import av\n        original_av_open = av.open\n        def patched_av_open(*args, **kwargs):\n            kwargs.pop("metadata_errors", None)\n            return original_av_open(*args, **kwargs)\n        av.open = patched_av_open\n    except Exception:\n        pass\n\n    try:\n        from faster_whisper import WhisperModel\n        import faster_whisper\n    except ImportError as ex:\n        parser.error("Missing faster-whisper. Run: python -m pip install faster-whisper\\n" + str(ex))\n\n    print("Source:", src.name, "| measured duration:", timestamp(duration), flush=True)\n    print("Loading multilingual model:", args.model, "(first use may download weights)", flush=True)\n    model = WhisperModel(args.model, device=args.device, compute_type=args.compute_type)\n\n    start = max(0.0, min(args.sample_start, max(0, duration - 1)))\n    length = min(args.sample_duration, duration - start)\n    if length < 1:\n        raise RuntimeError("Unable to choose a \u003E1s recording sample.")\n    sample_path = outdir / "_sample_test.wav"\n    run_cmd(["ffmpeg", "-y", "-v", "error", "-ss", str(start), "-i", str(src),\n             "-t", str(length), "-vn", "-ac", "1", "-ar", "16000", str(sample_path)])\n    try:\n        sample_iter, sample_info = transcribe(\n            model, sample_path, args.language, word_timestamps=False, beam_size=3)\n        sample_rows = [record(segment, i + 1) for i, segment in enumerate(sample_iter)]\n    finally:\n        sample_path.unlink(missing_ok=True)\n    sample_summary = {\n        "source_tested": src.name, "real_sample_start_seconds": start,\n        "real_sample_duration_seconds": length,\n        "detected_language": getattr(sample_info, "language", None),\n        "language_probability": getattr(sample_info, "language_probability", None),\n        "model": args.model, "segments": sample_rows,\n        "test_outcome": "recognisable_text" if any(row["text"] for row in sample_rows) else "no_text_detected"\n    }\n    dump_json(outdir / "sample_sanity.json", sample_summary)\n    print("\\nSHORT REAL-AUDIO SANITY TEST:", sample_summary["test_outcome"], flush=True)\n    for row in sample_rows[:8]:\n        print(f\u0027  {row["start_timestamp"]} {row["text"][:170]}\u0027, flush=True)\n    print("Language:", sample_summary["detected_language"],\n          "| confidence:", sample_summary["language_probability"], flush=True)\n    if not any(row["text"] for row in sample_rows):\n        raise RuntimeError(\n            "The sample returned no text. Examine sample_sanity.json and choose "\n            "a different --sample-start (e.g. 120) before full transcription.")\n    if args.sample_only:\n        print("\\nSample-only test complete; full lecture was NOT transcribed.", flush=True)\n        return\n\n    print("\\nTranscribing the WHOLE original recording ...", flush=True)\n    seg_iter, full_info = transcribe(\n        model, src, args.language, word_timestamps=True, beam_size=5)\n    segments = []\n    for seg in seg_iter:\n        segments.append(record(seg, len(segments) + 1))\n        if len(segments) % 40 == 0:\n            print("Processed", len(segments), "speech cues", flush=True)\n    if not segments:\n        raise RuntimeError("The full recording yielded zero speech cues. No transcript ZIP made.")\n    metadata = {\n        "transcription_status": "MACHINE GENERATED - HUMAN REVIEW REQUIRED",\n        "created_utc": dt.datetime.now(dt.timezone.utc).isoformat(),\n        "source_filename": src.name,\n        "source_size_bytes": src.stat().st_size,\n        "source_sha256": digest(src),\n        "measured_source": meta_probe,\n        "model": args.model,\n        "language_requested": args.language,\n        "detected_language": getattr(full_info, "language", None),\n        "detected_language_probability": getattr(full_info, "language_probability", None),\n        "model_device": args.device,\n        "model_compute_type": args.compute_type,\n        "python_version": sys.version,\n        "platform": platform.platform(),\n        "speech_cue_count": len(segments),\n        "flagged_cue_count": sum(int(r["review_required"]) for r in segments),\n        "total_recognised_characters": sum(len(r["text"]) for r in segments),\n        "original_audio_modified": False,\n        "note": "ASR cue timestamps are approximate; do not treat as final PPTX/MP4 timing."\n    }\n    write_outputs(outdir, src, segments, metadata)\n    package = outdir.parent / (outdir.name + "_Transcript_Review_Package.zip")\n    with zipfile.ZipFile(package, "w", compression=zipfile.ZIP_DEFLATED) as z:\n        for child in sorted(outdir.iterdir()):\n            if child.is_file() and not child.name.startswith("_"):\n                z.write(child, arcname=child.name)\n    print("\\nTRANSCRIPTION PACKAGE READY:", package, flush=True)\n    print("Cues:", len(segments), "| flagged:",\n          metadata["flagged_cue_count"], "| original audio not included in ZIP", flush=True)\n    print("Upload this ZIP back to your AI Atheer Mind conversation.", flush=True)\n\n\nif __name__ == "__main__":\n    main()\n', encoding='utf-8')
print('Script installed in this Colab session with compatibility monkeypatch.')

Script installed in this Colab session with compatibility monkeypatch.


In [11]:
# Verify the patch in the saved script file
with open('/content/transcribe_lecture.py', 'r') as f:
    lines = f.readlines()

# Display the first 40 lines of the script containing the patch
print("".join(lines[:40]))

#!/usr/bin/env python3
"""
AI Atheer Mind - free local / Google Colab timestamped transcription runner.
Source audio never leaves the selected runtime through this script.
Requires: Python 3.9+, FFmpeg/FFprobe, faster-whisper (downloads model weights
on first use). The model's transcripts are MACHINE GENERATED and MUST be reviewed.
"""
from __future__ import annotations

# MONKEYPATCH: Resolve PyAV v13+ TypeError: open() got an unexpected keyword argument 'metadata_errors'
try:
    import av
    original_av_open = av.open
    def patched_av_open(*args, **kwargs):
        kwargs.pop("metadata_errors", None)
        return original_av_open(*args, **kwargs)
    av.open = patched_av_open
except ImportError:
    pass

import argparse
import csv
import datetime as dt
import hashlib
import json
import os
import platform
import re
import shutil
import subprocess
import sys
import zipfile
from pathlib import Path


def run_cmd(args):
    result = subprocess.run(args, capture_output=True, text=T

In [13]:
# Step 4 — short-sample recognition test BEFORE processing the whole recording.
# language: 'auto' (detect), 'en' (English), 'ar' (Arabic).
# If the sample is silent, adjust sample_start_sec to an area with speech.
import sys, subprocess
model_size = "small"          # 'medium' may be better when a GPU is available
language = "auto"             # choose 'ar' or 'en' only if automatic detection is wrong
sample_start_sec = 120        # seconds into the REAL recording (updated to 120)
outroot = "/content/atheer_transcripts"
cmd = [sys.executable, "/content/transcribe_lecture.py",
       "--input", audio_path,
       "--output-dir", outroot,
       "--model", model_size,
       "--language", language,
       "--sample-start", str(sample_start_sec),
       "--sample-only"]

# Run the command
print("Running Step 4 sample transcription test...")
result = subprocess.run(cmd, capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print("--- SCRIPT ERROR LOGS ---", file=sys.stderr)
    print(result.stderr, file=sys.stderr)
    raise subprocess.CalledProcessError(result.returncode, cmd, result.stdout, result.stderr)
else:
    print("Review the sample lines above for recognizable words/language before continuing.")

Running Step 4 sample transcription test...
Source: Week_04_24m_Rescuing_Williwonk_s_Chocolates_with_Systems_Analys.m4a | measured duration: 00:23:26.293
Loading multilingual model: small (first use may download weights)

SHORT REAL-AUDIO SANITY TEST: recognisable_text
  00:00:00.000 story. But behind the scenes, their daily operations are completely melting down.
  00:00:04.640 Quite literally in some cases. Right. Now before we get into the messy details, I
  00:00:09.120 want to introduce a rule for today's deep dive. Oh yes, the notebook rule. Exactly.
  00:00:13.880 We really want you, the listener, to grasp these concepts. So whenever we
  00:00:18.240 introduce a critical technical term today, I'm gonna ask you to get out your
  00:00:21.640 notebook, write the term down and repeat it aloud three times. It sounds a
  00:00:26.120 bit like being back in school, but it genuinely helps cement the framework in
  00:00:29.640 your mind.
Language: en | confidence: 0.9994404911994934



In [12]:
import os
from pathlib import Path

out_path = Path('/content/atheer_transcripts')
if out_path.exists():
    print(f"[Found] Output directory: {out_path}")
    for p in out_path.glob('**/*'):
        if p.is_file():
            print(f"  - File: {p.relative_to(out_path)} ({p.stat().st_size} bytes)")
        elif p.is_dir():
            print(f"  - Dir:  {p.relative_to(out_path)}")
else:
    print(f"[Not Found] The directory {out_path} does not exist yet. Please make sure to successfully execute Step 4.")

[Found] Output directory: /content/atheer_transcripts
  - Dir:  Week_04_24m_Rescuing_Williwonk_s_Chocolates_with_Systems_Analys


In [14]:
# Step 5 — process the FULL recording and export VTT/SRT/TXT/JSON/CSV with timestamps.
# The script reruns the sample automatically before full inference, as a safety check.
import sys, subprocess
cmd = [sys.executable, "/content/transcribe_lecture.py",
       "--input", audio_path, "--output-dir", outroot,
       "--model", model_size, "--language", language,
       "--sample-start", str(sample_start_sec)]

print("Processing full transcription... This may take a few minutes.")
subprocess.run(cmd, check=True)
print("Full inference finished successfully! Review the transcript and flagged uncertainties.")

Processing full transcription... This may take a few minutes.
Full inference finished successfully! Review the transcript and flagged uncertainties.


In [15]:
# Step 6 — download the ZIP. Then upload it to the original AI Atheer Mind chat.
from google.colab import files
from pathlib import Path
packages = sorted(Path(outroot).glob("*_Transcript_Review_Package.zip"))
assert packages, "No completed transcript ZIP found. Finish Step 5 first."
print("Downloading:", packages[-1].name)
files.download(str(packages[-1]))


Downloading: Week_04_24m_Rescuing_Williwonk_s_Chocolates_with_Systems_Analys_Transcript_Review_Package.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

### What this ZIP contains and what it does NOT certify

`transcript.vtt`, `transcript.srt`, `transcript.txt`, `transcript.json`, `transcript.csv`, `run_metadata.json`, `sample_sanity.json`, `uncertainty_notes.txt`, `manifest.json` and `README_FIRST.txt`.

The original M4A is **not** included. The transcript is machine-generated. Audio-based slide segmentation, exact video transitions and full-lecture transcript quality still require review against the recording. Return the ZIP to AI Atheer Mind for those next production stages.
